# Review machine unlearning results

Run `python -m unlearning_lab.experiment` first, then use this notebook to inspect the retain-vs-forget tradeoff table.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

report_dir = Path("../reports")
metrics_path = report_dir / "method_metrics.csv"

if not metrics_path.exists():
    raise FileNotFoundError("Run the experiment first: python -m unlearning_lab.experiment")

metrics = pd.read_csv(metrics_path)
metrics


A useful method should keep retain accuracy high while moving forget-class confidence closer to exact retraining. Exact retraining is included as the reference, not as the cheap method.

In [ ]:
ranking = metrics.assign(
    total_gap=metrics["retain_accuracy_gap"] + metrics["forget_confidence_gap"] + metrics["forget_accuracy_gap"]
).sort_values("total_gap")

ranking[[
    "method",
    "test_retain_accuracy",
    "test_forget_accuracy",
    "test_forget_confidence",
    "runtime_seconds",
    "total_gap",
]]


In [ ]:
axis = metrics.plot.scatter(
    x="test_forget_confidence",
    y="test_retain_accuracy",
    s=90,
    figsize=(7, 5),
)
for row in metrics.to_dict("records"):
    axis.annotate(
        row["method"],
        (row["test_forget_confidence"], row["test_retain_accuracy"]),
        xytext=(5, 4),
        textcoords="offset points",
        fontsize=9,
    )
axis.set_title("Retain utility vs forgetting")
axis.set_xlabel("forget-class confidence")
axis.set_ylabel("retain-class accuracy")
axis.grid(alpha=0.25)
plt.tight_layout()
